In [13]:
import glob
import os
import re

import dask.array as da
import napari
from natsort import natsorted
from skimage.measure import label
from tqdm.auto import tqdm


def extract_metadata(basename):
    """
    Extracts replicate number, mouse identifier, and label from a Zarr basename.

    Args:
        basename (str): The Zarr filename (e.g., 'rep1_mouse6_bot.zarr').

    Returns:
        tuple: (replicate_num, mouse_num, label)
    """
    # Regex pattern:
    # (rep\d+): Group 1 captures 'rep' followed by one or more digits (replicate number)
    # (mouse\d+): Group 2 captures 'mouse' followed by one or more digits (mouse identifier)
    # ([a-z]+): Group 3 captures one or more lowercase letters (label: bot, top, etc.)
    pattern = r'rep(\d+)_mouse(\d+)_([a-z]+)\.zarr'
    
    match = re.search(pattern, basename)
    
    if match:
        return match.groups()
    else:
        # Return None or raise an error for cases that don't match the expected format
        return (None, None, None)



In [20]:
zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_*/zarr/*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if ('top' in fn) or ('bot' in fn) or ('left' in fn) or ('right' in fn)])

for zarr_address in tqdm(zarr_addresses):
    # find zarr dir with labels subdir
    zarr_dir = os.path.dirname(zarr_address)
    og_zarr_fns = glob.glob(os.path.join(zarr_dir, '*.zarr'))
    og_zarr_fn = [fn for fn in og_zarr_fns if ('corrupt' not in fn) and \
                                               ('notebook' not in fn) and \
                                               ('top' not in fn) and \
                                               ('bot' not in fn) and \
                                               ('left' not in fn) and \
                                               ('right' not in fn)][0]
    
    print(og_zarr_fn)

    # print(f"Processing: {os.path.basename(zarr_address)}")
    
    # 2. EXTRACT CROP LABEL (e.g., 'bot')
    basename = os.path.basename(zarr_address)
    replicate_num, mouse_num, label = extract_metadata(basename)

    # 3. DEFINE SOURCE AND DESTINATION PATHS
    # The source labels directory in the OG Zarr (assumes labels are stored like: og_zarr_fn/labels/ground_truth_bot)
    source_labels_dir = os.path.join(og_zarr_fn, f'labels/ground_truth_{label}')
    
    # The destination directory *inside* the new cropped Zarr file
    new_labels_dir_parent = os.path.join(zarr_address, 'labels')
    destination_labels_dir = os.path.join(new_labels_dir_parent, f'ground_truth_{label}')
    print(destination_labels_dir)
    # # 4. EXECUTE COPY
    # if os.path.exists(source_labels_dir):
    #     # Ensure the parent 'labels' directory exists in the new Zarr
    #     os.makedirs(new_labels_dir_parent, exist_ok=True)

    #     # Remove the destination if it already exists to allow copytree to work
    #     if os.path.exists(destination_labels_dir):
    #         shutil.rmtree(destination_labels_dir)
            
    #     # The core operation: copy the entire directory structure
    #     shutil.copytree(source_labels_dir, destination_labels_dir)
    #     # print(f"Copied labels from {os.path.basename(og_zarr_fn)} to {os.path.basename(zarr_address)}")
    # else:
    #     # This occurs if segmentation was not yet performed on the OG Zarr for this crop
    #     print(f"Warning: Source labels not found at {source_labels_dir}")
    print()
print("Label copying complete.")

  0%|          | 0/12 [00:00<?, ?it/s]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/rep1_mouse6_bot.zarr/labels/ground_truth_bot

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556.zarr
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/rep1_mouse6_top.zarr/labels/ground_truth_top

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/zarr/rep1_mouse7_bot.zarr/labels/ground_truth_bot

/mnt

In [21]:
zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_*/zarr/*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if ('top' in fn) or ('bot' in fn) or ('left' in fn) or ('right' in fn)])

results = []

for zarr_address in tqdm(zarr_addresses):

    # 2. EXTRACT CROP LABEL (e.g., 'bot')
    basename = os.path.basename(zarr_address)
    replicate_num, mouse_num, label = extract_metadata(basename)
    # Load data
    images = da.from_zarr(f"{zarr_address}/s0")
    masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_{label}/0")
    break

  0%|          | 0/12 [00:00<?, ?it/s]

In [22]:
images

dask.array<from-zarr, shape=(3, 35221, 44308), dtype=>u2, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

In [24]:
masks

dask.array<from-zarr, shape=(54144, 45850), dtype=uint8, chunksize=(1692, 1433), chunktype=numpy.ndarray>

In [ ]:
viewer = napari.Viewer(title = 'testing alignment')
viewer.add_image(images, channel_axis=0)
viewer.add_labels(masks)